# Advanced RAG Retrieval Evaluation using Query Expansion

## Introduction

Retrieval-Augmented Generation (RAG) systems enhance the capabilities of Large Language Models (LLMs) by retrieving relevant information from external knowledge sources before generating responses. However, traditional similarity-based retrieval often struggles with complex financial and analytical questions because users may phrase queries differently from the terminology used in source documents.

This notebook implements and evaluates a Query Expansion-based retrieval improvement technique on Tesla Form 10-K annual reports. The objective is to improve evidence retrieval quality by generating multiple semantically related search queries using a Large Language Model and comparing the results against a baseline retrieval approach.

The solution follows the requirements outlined in the Advanced RAG Retrieval Evaluation assignment and leverages the Groq API, vector embeddings, and a Chroma vector database to build a reproducible retrieval pipeline. Retrieved evidence is then used to generate grounded answers supported by citations and metadata.

## Objectives

### Primary Objectives

1. Build a baseline RAG retrieval pipeline using the original user query.
2. Generate multiple query variants using an LLM-based query expansion strategy.
3. Retrieve relevant document chunks for each expanded query.
4. Combine and deduplicate retrieval results from multiple query perspectives.
5. Generate evidence-based answers using retrieved Tesla 10-K content.
6. Compare baseline retrieval performance against query-expanded retrieval.

### Query Expansion Goals

* Improve retrieval recall by discovering semantically related content.
* Capture financial, operational, strategic, and risk-related terminology variations.
* Reduce missed evidence caused by vocabulary mismatches between user questions and source documents.
* Maintain retrieval relevance while increasing document coverage.

### Evaluation Goals

* Analyze retrieval quality for benchmark financial questions.
* Compare retrieved evidence across baseline and expanded approaches.
* Identify improvements in answer completeness and citation quality.
* Evaluate trade-offs between retrieval recall and precision.
* Document failure modes and production considerations for enterprise RAG systems.

## Workflow

1. Load and preprocess Tesla 10-K documents.
2. Create document chunks and preserve metadata.
3. Generate embeddings and store chunks in ChromaDB.
4. Implement baseline similarity search retrieval.
5. Generate expanded search queries using Groq.
6. Retrieve and fuse results from expanded queries.
7. Generate answers using retrieved evidence.
8. Compare baseline and improved retrieval performance.
9. Export results and perform analytical evaluation.

This notebook demonstrates how retrieval-focused prompt engineering can significantly improve the effectiveness of RAG systems when working with complex financial documents and analyst-style questions.




# Import Required Libraries

This section imports all required Python libraries for the Retrieval-Augmented Generation (RAG) pipeline. The imported packages support document loading, text chunking, embeddings generation, vector database operations, metadata handling, and interaction with the Groq API.


In [ ]:

import os
import time
from datetime import datetime

import chromadb
from langchain_chroma import Chroma

from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader

from IPython.display import display, Markdown


# Load Environment Variables and Initialize Groq Client

The Groq API client is initialized using credentials stored in environment variables. This approach keeps API keys secure and prevents sensitive information from being hardcoded into the notebook.


In [3]:
# Load Environment Variables & Initialize Groq Client

from dotenv import load_dotenv
from groq import Groq

load_dotenv()

# Load API Key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in .env file")

# Initialize Client
client = Groq(api_key=GROQ_API_KEY)

print("Groq Client Initialized Successfully")

Groq Client Initialized Successfully


# Define Project Configuration

This section defines the core configuration variables used throughout the project, including the LLM model, embedding model, vector database location, document source directory, chunking parameters, and retrieval settings.


In [10]:
MODEL_NAME = "openai/gpt-oss-120b"

EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"

CHROMA_DB_PATH = "./tesla_db"
PDF_FOLDER = "tesla-annual-reports"

COLLECTION_NAME = "tesla-10k-2019-to-2023"

CHUNK_SIZE = 512
CHUNK_OVERLAP = 16

TOP_K = 2

## Initialize Embedding Model

This step loads the embedding model used to convert document chunks and user queries into vector representations for semantic similarity search.

In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

print("Embeddings Ready")

Embeddings Ready


## Load Tesla Annual Reports

Tesla annual report documents are loaded from the local directory. These documents form the knowledge base used for retrieval and question answering.

In [54]:
# Load Tesla Annual Reports

start_time = time.time()

pdf_loader = PyPDFDirectoryLoader(PDF_FOLDER)

documents = pdf_loader.load()

print(f"Documents Loaded for the chunk: {len(documents)}")
print(f"Time Taken: {time.time()-start_time:.2f} sec")

Documents Loaded for the chunk: 1183
Time Taken: 189.50 sec


## Chunk Documents for Retrieval

The loaded documents are divided into overlapping chunks to improve retrieval precision while preserving contextual information across chunk boundaries.

In [55]:
# Chunk Documents

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

tesla_10k_chunks = text_splitter.split_documents(
    documents
)

print(f"Total Chunks: {len(tesla_10k_chunks):,}")

Total Chunks: 3,337


## Analyze Chunk Statistics

Basic statistics such as average, minimum, and maximum chunk sizes are calculated to validate the chunking strategy.

In [56]:
chunk_lengths = [
    len(doc.page_content)
    for doc in tesla_10k_chunks
]

print(f"The average chunk Size : {sum(chunk_lengths)/len(chunk_lengths):.0f}")
print(f"The minimum chunk Size : {min(chunk_lengths)}")
print(f"The maximum chunk Size : {max(chunk_lengths)}")

The average chunk Size : 1184
The minimum chunk Size : 2
The maximum chunk Size : 1917


## Inspect Sample Chunk

A sample chunk is displayed to verify that document content has been loaded and segmented correctly.

In [15]:
print(tesla_10k_chunks[0].page_content)

UNITED	STATES
SECURITIES	AND	EXCHANGE	COMMISSION
Washington,	D.C.	20549
	
FORM	
10-K/A
(Amendment	No.	1)
	
(Mark	One)
☒
ANNUAL	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	fiscal	year	ended	
December	31,	
2021
	
OR
☐
TRANSITION	REPORT	PURSUANT	TO	SECTION	13	OR	15(d)	OF	THE	SECURITIES	EXCHANGE	ACT	OF	1934
For	the	transition	period	from	
																				
	to	
																					
Commission	File	Number:	
001-34756
	
Tesla,	Inc.
(Exact	name	of	registrant	as	specified	in	its	charter)
	
	
Delaware
	
91-2197729
(State	or	other	jurisdiction	of
incorporation	or	organization)
	
(I.R.S.	Employer
Identification	No.)
	
	
	
1	Tesla	Road
Austin
,	
Texas
	
78725
(Address	of	principal	executive	offices)
	
(Zip	Code)
(
512
)	
516-8177
(Registrant’s	telephone	number,	including	area	code)
Securities	registered	pursuant	to	Section	12(b)	of	the	Act:
	
Title	of	each	class
Trading	Symbol(s)
Name	of	each	exchange	on	which	registered
Common	stock
TSLA
The	Nasdaq	Gl

## Create and Populate Chroma Vector Database

Document chunks are embedded and stored in a persistent Chroma vector database for efficient semantic retrieval.

In [57]:
tesla_10k_collection = 'tesla-10k-2019-to-2023'
chromadb_client = chromadb.PersistentClient(
    path="./tesla_db"
)

vectorstore = Chroma(
    collection_name=tesla_10k_collection,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory="./tesla_db"
)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


## Initialize ChromaDB Connection

A connection to the persistent Chroma database is established for future retrieval operations.

In [58]:
# ChromaDB Initialization

chromadb_client = chromadb.PersistentClient(
    path=CHROMA_DB_PATH
)

print("ChromaDB Connected")
print(f"The Path of database: {CHROMA_DB_PATH}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


ChromaDB Connected
The Path of database: ./tesla_db


## Verify ChromaDB Client

The initialized ChromaDB client object is displayed to confirm successful database connectivity.

In [18]:
chromadb_client

## Load Persisted Vector Store

The previously created vector database is loaded from disk to enable retrieval without rebuilding embeddings.

In [59]:
# Load Persisted Vector Store

vectorstore_persisted = Chroma(
    collection_name=COLLECTION_NAME,
    collection_metadata={"hnsw:space": "cosine"},
    embedding_function=embedding,
    client=chromadb_client,
    persist_directory=CHROMA_DB_PATH
)

collection = chromadb_client.get_collection(
    COLLECTION_NAME
)

print("The vector store has been loaded from disk.")
print(f"The collection name of the vector store is: {COLLECTION_NAME}")
print(f"The number of chunks in the collection is: {collection.count():,}")

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


The vector store has been loaded from disk.
The collection name of the vector store is: tesla-10k-2019-to-2023
The number of chunks in the collection is: 3,337


## Validate Indexed Documents

The total number of indexed chunks is checked to ensure successful document ingestion.

In [20]:
collection = vectorstore_persisted._collection
total_docs = collection.count()
print(total_docs)

3337


## Configure Retriever

A similarity-based retriever is created to fetch the most relevant document chunks for a given query.

In [60]:
# Retriever Configuration

retriever = vectorstore_persisted.as_retriever(
    search_type="similarity",
    search_kwargs={"k": TOP_K}
)

print("The retriever has been initialized with similarity search.")
print(f"The search type is: similarity")
print(f"The top K documents are: {TOP_K}")

The retriever has been initialized with similarity search.
The search type is: similarity
The top K documents are: 2


## Inspect Retriever Configuration

The retriever object is displayed for verification and debugging purposes.

In [22]:
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000022B17EAA540>, search_kwargs={'k': 2})

## Define Question Answering Prompt

A system prompt is created to guide the language model in generating grounded financial answers based only on retrieved evidence.

In [61]:
qna_system_message = """
You are an expert financial analyst specializing in corporate annual reports,
SEC filings, risk disclosures, financial performance analysis, business strategy,
operations, and investor communications.

Your task is to answer questions using ONLY the information provided in the context.

RULES:

1. Use ONLY the provided context to generate your answer.
2. Do NOT use prior knowledge, assumptions, or external information.
3. If the context does not contain enough information to answer the question,
   respond exactly:
   "I don't know."
4. Be factual, precise, and evidence-based.
5. Preserve all numerical values, percentages, dates, monetary amounts,
   growth rates, and quantitative information exactly as stated.
6. When discussing financial performance, clearly mention the relevant fiscal year(s).
7. When information spans multiple years, compare them accurately using the evidence provided.
8. If multiple pieces of evidence support the answer, synthesize them into a coherent response.
9. Do NOT speculate, infer missing facts, or provide opinions.
10. Do NOT mention:
    - Context
    - Documents
    - Retrieved chunks
    - Sources
    - Embeddings
    - Vector databases
    - Retrieval systems
11. If the question asks for comparison, explain similarities, differences,
    trends, and supporting evidence.
12. If risks, opportunities, or strategic priorities are discussed, summarize
    them objectively based on the evidence.
13. Keep answers concise but sufficiently detailed to fully address the question.

ANSWER FORMAT:

For analytical questions:
- Start with a direct answer.
- Follow with supporting evidence.
- Mention relevant years where applicable.

For comparison questions:
- Provide a comparison summary.
- Highlight key differences and similarities.
- Support statements with evidence from the provided information.

For risk-related questions:
- Identify the major risks.
- Explain their potential impact.
- Include any relevant operational or financial context.

Never include information that cannot be directly supported by the provided context.
"""


qna_user_message_template = """
You are provided with relevant excerpts from Tesla annual reports.

==================== CONTEXT ====================

{context}

=================================================

QUESTION:

{question}

INSTRUCTIONS:

- Use only the information contained in the context.
- Preserve numerical values exactly.
- Mention relevant fiscal years whenever available.
- Do not make assumptions or use outside knowledge.
- If the answer is not supported by the context, respond exactly:
  "I don't know."

ANSWER:
"""

## Retrieve Relevant Documents

A sample financial query is executed against the retriever to obtain the most relevant document chunks.

In [62]:
# Retrieve Relevant Documents

user_query = "What was the automotive revenue in 2021?"

start_time = time.time()

relevant_document_chunks = retriever.invoke(user_query)

end_time = time.time()

print(f"The query of the user is: {user_query}")
print(f"The number of retrieved document chunks is: {len(relevant_document_chunks)}")
print(f"The retrieval time is: {end_time-start_time:.2f} sec")

The query of the user is: What was the automotive revenue in 2021?
The number of retrieved document chunks is: 2
The retrieval time is: 0.08 sec


## Inspect Retrieved Document

The first retrieved document chunk is displayed to verify retrieval quality.

In [25]:
for document in relevant_document_chunks:
    print(document)
    break


page_content='2022
2021
$
%
$
%
Automotive	sales
$
78,509	
$
67,210	
$
44,125	
$
11,299	
17	
%
$
23,085	
52	
%
Automotive	regulatory	credits
1,790	
1,776	
1,465	
14	
1	
%
311	
21	
%
Automotive	leasing
2,120	
2,476	
1,642	
(356)
(14)
%
834	
51	
%
Total	automotive	revenues
82,419	
71,462	
47,232	
10,957	
15	
%
24,230	
51	
%
Services	and	other
8,319	
6,091	
3,802	
2,228	
37	
%
2,289	
60	
%
Total	automotive	&	services	and	other	segment
revenue
90,738	
77,553	
51,034	
13,185	
17	
%
26,519	
52	
%
Energy	generation	and	storage	segment	revenue
6,035	
3,909	
2,789	
2,126	
54	
%
1,120	
40	
%
Total	revenues
$
96,773	
$
81,462	
$
53,823	
$
15,311	
19	
%
$
27,639	
51	
%
Automotive	&	Services	and	Other	Segment
Automotive	sales	revenue	includes	revenues	related	to	cash	and	financing	deliveries	of	new	Model	S,	Model	X,	Semi,	Model	3,	Model	Y,	and
Cybertruck	vehicles,	including	access	to	our	FSD	Capability	features	and	their	ongoing	maintenance,	internet	connectivity,	free	Supercharging	programs
and	ov

## Display Retrieved Context

All retrieved chunks are printed for detailed inspection and validation.

In [26]:
i = 0
for document in relevant_document_chunks:
    print(f"-------Chunk {i}-------")
    print(document.page_content.replace("\t", " "))

    i += 1

-------Chunk 0-------
2022
2021
$
%
$
%
Automotive sales
$
78,509 
$
67,210 
$
44,125 
$
11,299 
17 
%
$
23,085 
52 
%
Automotive regulatory credits
1,790 
1,776 
1,465 
14 
1 
%
311 
21 
%
Automotive leasing
2,120 
2,476 
1,642 
(356)
(14)
%
834 
51 
%
Total automotive revenues
82,419 
71,462 
47,232 
10,957 
15 
%
24,230 
51 
%
Services and other
8,319 
6,091 
3,802 
2,228 
37 
%
2,289 
60 
%
Total automotive & services and other segment
revenue
90,738 
77,553 
51,034 
13,185 
17 
%
26,519 
52 
%
Energy generation and storage segment revenue
6,035 
3,909 
2,789 
2,126 
54 
%
1,120 
40 
%
Total revenues
$
96,773 
$
81,462 
$
53,823 
$
15,311 
19 
%
$
27,639 
51 
%
Automotive & Services and Other Segment
Automotive sales revenue includes revenues related to cash and financing deliveries of new Model S, Model X, Semi, Model 3, Model Y, and
Cybertruck vehicles, including access to our FSD Capability features and their ongoing maintenance, internet connectivity, free Supercharging program

## Define User Query

The evaluation query is stored in a variable for reuse across retrieval and generation stages.

In [63]:
user_query = "What was the automotive revenue in 2021?"

## Generate Baseline RAG Answer

Retrieved context is combined with the prompt and sent to the language model to generate an answer.

In [64]:

model_name = 'openai/gpt-oss-120b'

relevant_document_chunks = retriever.invoke(user_query)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = "\n---\n".join(context_list)

prompt = [
    {'role': 'system', 'content': qna_system_message},
    {'role': 'user', 'content': qna_user_message_template.format(
         context=context_for_query,
         question=user_query
        )
    }
]


try:
    response = client.chat.completions.create(
        model=model_name,
        messages=prompt,
        temperature=0
    )

    prediction = response.choices[0].message.content.strip()
except Exception as e:
    prediction = f'Sorry, I encountered the following error: \n {e}'

print(prediction)

The automotive revenue for 2021 was **$47,232**.  

*Evidence:* In the “Results of Operations” table, the row labeled “Total automotive revenues” lists $47,232 for the year ended December 31, 2021.


## Define Assignment Benchmark Questions

The benchmark questions provided in the assignment are stored for systematic evaluation of retrieval performance.

In [65]:
# ASSIGNMENT QUESTIONS


questions= {
    "Q1": "Does Tesla's growth story appear more constrained by external supply risk or internal execution and cost structure? Use evidence across Risk Factors and MD&A.",

    "Q2": "Explain how Tesla's AI and product roadmap is reflected in spending, operational priorities, and risk disclosures. Avoid generic AI claims.",

    "Q3": "A supplier asks whether Tesla is exposed to concentration risk across factories, suppliers, raw materials, or geographies. Prepare a concise risk note with citations.",

    "Q4": "Compare the strategic importance of automotive operations and energy generation/storage using evidence from the 10-K. What disclosures are needed to support a business recommendation?"
}

## Create Query Expansion Prompt

A prompt template is designed to generate multiple retrieval-focused query variations that preserve user intent.

In [66]:
# Query Expansion Prompt

query_expansion_system_message = """
You are a financial analyst specializing in SEC 10-K filings.

Generate 6 retrieval-oriented query variants.

The variants must cover:

1. Financial analyst phrasing
2. Risk factor phrasing
3. Operational phrasing
4. Strategy phrasing
5. Synonym-based phrasing
6. Subtopic decomposition

Rules:

- Preserve original intent
- Do not introduce unrelated topics
- Do not answer the question
- Return exactly 6 queries
- One query per line
- No numbering
- No bullets
"""

query_expansion_user_template = """
Original Question:
{question}
"""

## Build Query Expansion Request

The prompt structure for query expansion is assembled using one of the benchmark questions.

In [67]:
prompt = [
    {'role':'system','content':query_expansion_system_message},
    {'role':'user','content':query_expansion_user_template.format(
        question=questions["Q1"]
    )}
]

## Inspect Query Expansion Prompt

The constructed prompt is displayed before sending it to the language model.

In [68]:
prompt

[{'role': 'system',
  'content': '\nYou are a financial analyst specializing in SEC 10-K filings.\n\nGenerate 6 retrieval-oriented query variants.\n\nThe variants must cover:\n\n1. Financial analyst phrasing\n2. Risk factor phrasing\n3. Operational phrasing\n4. Strategy phrasing\n5. Synonym-based phrasing\n6. Subtopic decomposition\n\nRules:\n\n- Preserve original intent\n- Do not introduce unrelated topics\n- Do not answer the question\n- Return exactly 6 queries\n- One query per line\n- No numbering\n- No bullets\n'},
 {'role': 'user',
  'content': "\nOriginal Question:\nDoes Tesla's growth story appear more constrained by external supply risk or internal execution and cost structure? Use evidence across Risk Factors and MD&A.\n"}]

## Create Query Expansion Prompt Builder

A reusable helper function is implemented to generate query expansion prompts dynamically.

In [69]:
def build_query_expansion_prompt(question):
    return [
        {
            "role": "system",
            "content": query_expansion_system_message
        },
        {
            "role": "user",
            "content": query_expansion_user_template.format(
                question=question
            )
        }
    ]

## Generate Expanded Queries

The language model produces multiple alternative search queries to improve retrieval coverage.

In [70]:
# Query Expansion

def expand_query(question):

    prompt = build_query_expansion_prompt(question)

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=prompt,
        temperature=0
    )

    return [
        q.strip()
        for q in response.choices[0].message.content.split("\n")
        if q.strip()
    ]

## Create Generic LLM Response Helper

A reusable helper function is implemented for interacting with the Groq language model.

In [71]:
# LLM Response Helper

def generate_response(messages, temperature=0):

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=temperature
    )

    return response.choices[0].message.content.strip()

## Retrieve Documents Using Query Expansion

Documents are retrieved for each expanded query and aggregated for further processing.

In [72]:
def retrieve_with_expansion(question):

    expanded_queries = expand_query(question)

    retrieval_results = {}

    for query in expanded_queries:

        docs = vectorstore_persisted.similarity_search_with_score(
            query,
            k=TOP_K
        )

        retrieval_results[query] = docs

    fused_results = reciprocal_rank_fusion(
        retrieval_results
    )

    return expanded_queries, fused_results

## Collect Expanded Retrieval Results

Retrieved chunks from multiple query variants are gathered into a combined context set.

In [74]:
# Retrieve Documents Using Expanded Queries

expanded_context_list = []

for query in questions.values():

    docs = retriever.invoke(query)

    expanded_context_list.extend(
        [doc.page_content for doc in docs]
    )

print(f"The total number of retrieved document chunks is: {len(expanded_context_list)}")

The total number of retrieved document chunks is: 8


## Deduplicate Retrieved Chunks

Duplicate chunks are removed to ensure diverse and efficient context construction.

In [75]:
final_context_documents = list(
    dict.fromkeys(expanded_context_list)
)

print(
    f"The number of unique document chunks is: {len(final_context_documents)}"
)

The number of unique document chunks is: 6


## Define Financial Analyst QA Prompt

A more specialized prompt is created to generate grounded financial analysis using retrieved evidence.

In [76]:

qna_system_message = """
You are a senior financial analyst reviewing Tesla 10-K filings.

Answer the question using ONLY the information provided in the context.

Instructions:

1. Use only facts explicitly supported by the context.
2. Do not use outside knowledge, assumptions, or speculation.
3. If the available evidence is insufficient to answer the question,
   respond exactly:
   "I don't know."
4. Provide a concise, analytical, and evidence-based answer.
5. When discussing risks, strategy, operations, or financial performance:
   - synthesize evidence across multiple sections when available
   - explain relationships between facts and disclosures
   - highlight relevant trends, dependencies, or trade-offs
   - avoid unsupported interpretations or conclusions
6. Preserve numerical values, percentages, dates, and monetary figures exactly as provided.
7. Mention relevant fiscal year(s) whenever available.
8. Prioritize the most directly relevant evidence when multiple facts are available.
9. If multiple chunks support a statement, combine the evidence into a single coherent explanation.
10. Cite every major claim using the provided chunk identifiers.

Citation format:

[chunk_id]

Example:

Tesla's automotive revenue increased substantially due to higher vehicle deliveries and production scale. [expanded_2]

If multiple chunks support a statement:

[expanded_2, expanded_5]

Do not mention:
- context
- retrieved documents
- source passages
- chunks except in citations
- vector database
- retrieval process
- embeddings

Answer directly, professionally, and with a financial analyst's perspective.
"""

qna_user_message_template = """
<Context>
{context}
</Context>

<Question>
{question}
</Question>

Provide the response in the following format:

Answer:
<direct analytical answer>

Key Supporting Evidence:
- <evidence point 1>
- <evidence point 2>
- <evidence point 3>

Citations:
[list of supporting chunk identifiers]
"""


## Build QA Prompt Dynamically

A helper function is implemented to assemble question-answering prompts using context and user questions.

In [77]:
# QA Prompt Builder

def build_qa_prompt(context, question):

    return [
        {
            "role": "system",
            "content": qna_system_message
        },
        {
            "role": "user",
            "content": qna_user_message_template.format(
                context=context,
                question=question
            )
        }
    ]

## Reusable LLM Response Function

A reusable generation function is defined for answer synthesis tasks.

In [79]:
# LLM Response Helper

def generate_response(messages, temperature=0):

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=temperature
    )

    return response.choices[0].message.content.strip()

## Answer Questions Using Retrieved Context

Retrieved evidence is combined and sent to the language model to generate final answers.

In [80]:
def answer_question(
    questions,
    context_documents
):

    context = "\n\n---\n\n".join(
        context_documents
    )

    prompt = build_qa_prompt(
        context=context,
        question=questions
    )

    return generate_response(
        prompt,
        temperature=0
    )

## Verify Answer Function

The question-answering function is displayed to confirm successful implementation.

In [81]:
print(answer_question)

<function answer_question at 0x0000022B1AC162A0>


## Generate Sample Query Expansion Answer

A benchmark question is answered using the expanded retrieval pipeline.

In [82]:
prediction = answer_question(
    questions=questions["Q1"],
    context_documents=final_context_documents
)
print(prediction)

Answer:
Tesla’s growth appears more constrained by internal execution and cost‑structure challenges rather than external supply risks. The company emphasizes the need to scale manufacturing, improve delivery and service operations, and reduce costs, while the risk discussion focuses on limited operating history and the difficulty of meeting production and sales expectations—issues that are largely internal.

Key Supporting Evidence:
- Tesla notes that maintaining confidence is challenged by “any delays we may experience in scaling manufacturing, delivery and service operations to meet demand” and by “our quarterly production and sales performance compared with market expectations.” [22]
- The MD&A states that the company is “currently focused on increasing vehicle production, capacity and delivery capabilities, reducing costs, improving and developing our vehicles and battery technologies, vertically integrating…” indicating internal execution and cost efficiency are central to its gro

## Inspect Query Expansion Template

The query expansion template is displayed for validation and refinement.

In [83]:
print(query_expansion_user_template)


Original Question:
{question}



## Analyze Retrieval Improvements

A comparison function is implemented to evaluate improvements between baseline and expanded retrieval.

In [84]:
def retrieval_improvement_analysis(
    baseline_chunks,
    expanded_chunks
):

    baseline_sections = set(
        chunk.get("section", "Unknown")
        for chunk in baseline_chunks
    )

    expanded_sections = set(
        chunk.get("section", "Unknown")
        for chunk in expanded_chunks
    )

    new_sections = (
        expanded_sections - baseline_sections
    )

    analysis = (
        f"Baseline retrieved {len(baseline_chunks)} chunks "
        f"across {len(baseline_sections)} sections. "
        f"Expanded retrieval returned {len(expanded_chunks)} chunks "
        f"across {len(expanded_sections)} sections."
    )

    if new_sections:
        analysis += (
            f" Additional coverage obtained from sections: "
            f"{', '.join(new_sections)}."
        )

    return analysis

## Evaluate Retrieval Quality

A scoring function is defined to assess retrieval effectiveness and evidence quality.

In [85]:
def evaluate_quality(chunks):

    if not chunks:
        return "Low"

    scores = []

    for chunk in chunks:

        if isinstance(chunk, dict) and "score" in chunk:
            scores.append(chunk["score"])

    if len(scores) == 0:

        if len(chunks) >= 5:
            return "High"

        elif len(chunks) >= 3:
            return "Medium"

        else:
            return "Low"

    avg_score = sum(scores) / len(scores)

    if avg_score >= 0.80:
        return "High"

    elif avg_score >= 0.60:
        return "Medium"

    return "Low"

## Implement Baseline and Expanded Retrieval Evaluation

This section builds the evaluation framework required by the assignment, including baseline retrieval, expanded retrieval, answer generation, and result collection.

In [ ]:
import json

def retrieve_baseline(query, k=5):

    docs = vectorstore_persisted.similarity_search_with_score(
        query,
        k=k
    )

    results = []

    for doc, score in docs:

        results.append({
            "chunk_id": doc.metadata.get("chunk_id", ""),
            "section": doc.metadata.get("section", ""),
            "year": doc.metadata.get("year", ""),
            "score": float(score),
            "content": doc.page_content
        })

    return results


def retrieve_expanded(expanded_queries, k=5):

    all_chunks = {}

    for query in expanded_queries:

        docs = vectorstore_persisted.similarity_search_with_score(
            query,
            k=k
        )

        for doc, score in docs:

            chunk_id = doc.metadata.get(
                "chunk_id",
                hash(doc.page_content)
            )

            if (
                chunk_id not in all_chunks
                or score < all_chunks[chunk_id]["score"]
            ):
                all_chunks[chunk_id] = {
                    "chunk_id": chunk_id,
                    "section": doc.metadata.get("section", ""),
                    "year": doc.metadata.get("year", ""),
                    "score": float(score),
                    "content": doc.page_content
                }

    results = list(all_chunks.values())

    results.sort(key=lambda x: x["score"])

    return results


def create_citations(chunks):

    citations = []

    for chunk in chunks:

        citations.append({
            "chunk_id": chunk.get("chunk_id"),
            "section": chunk.get("section"),
            "year": chunk.get("year")
        })

    return citations


def evaluate_quality(chunks):

    if len(chunks) == 0:
        return {
            "retrieved_chunks": 0,
            "avg_score": None
        }

    scores = [
        chunk["score"]
        for chunk in chunks
    ]

    return {
        "retrieved_chunks": len(chunks),
        "avg_score": sum(scores) / len(scores)
    }


def retrieval_improvement_analysis(
    baseline_chunks,
    expanded_chunks
):

    baseline_ids = {
        chunk["chunk_id"]
        for chunk in baseline_chunks
    }

    expanded_ids = {
        chunk["chunk_id"]
        for chunk in expanded_chunks
    }

    new_chunks = expanded_ids - baseline_ids

    return {
        "baseline_count": len(baseline_ids),
        "expanded_count": len(expanded_ids),
        "new_chunks_found": len(new_chunks),
        "new_chunk_ids": list(new_chunks)
    }


all_results = []

for qid, question in questions.items():

    print(f"Processing {qid}...")

    expanded_queries = expand_query(question)

    baseline_chunks = retrieve_baseline(
        question
    )

    expanded_chunks = retrieve_expanded(
        expanded_queries
    )

    final_answer = answer_question(
        questions=question,
        context_documents=[
            chunk["content"]
            for chunk in expanded_chunks[:3]
        ]
    )

    result = {

        "question_id": qid,

        "original_query": question,

        "expanded_queries": expanded_queries,

        "baseline_top_chunks":
            baseline_chunks[:5],

        "expanded_top_chunks":
            expanded_chunks[:5],

        "final_answer":
            final_answer,

        "citations":
            create_citations(
                expanded_chunks[:5]
            ),

        "baseline_quality":
            evaluate_quality(
                baseline_chunks
            ),

        "expanded_quality":
            evaluate_quality(
                expanded_chunks
            ),

        "retrieval_improvement_analysis":
            retrieval_improvement_analysis(
                baseline_chunks,
                expanded_chunks
            )
    }

    all_results.append(result)

print("Completed all questions.")

# Save results
with open(
    "query_expansion_results.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_results,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Results saved.")

Processing Q1...
Processing Q2...
Processing Q3...
Processing Q4...
Completed all questions.
Results saved.


## Save Evaluation Results

All retrieval and generation outputs are saved in JSON format for submission and further analysis.

In [87]:
with open(
    "query_expansion_results.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_results,
        f,
        indent=4,
        ensure_ascii=False
    )

print("The results of the query expansion and retrieval improvement analysis have been saved to 'query_expansion_results.json'.")

The results of the query expansion and retrieval improvement analysis have been saved to 'query_expansion_results.json'.


## Verify Output Structure

The structure of the generated results is inspected to ensure compliance with the required schema.

In [88]:
print(all_results[0].keys())

dict_keys(['question_id', 'original_query', 'expanded_queries', 'baseline_top_chunks', 'expanded_top_chunks', 'final_answer', 'citations', 'baseline_quality', 'expanded_quality', 'retrieval_improvement_analysis'])


## Build Comparative Analysis Table

A comparison table is generated to evaluate baseline and improved retrieval performance across benchmark questions.

In [89]:
comparison_table = []

for result in all_results:

    baseline_count = len(
        result.get(
            "baseline_top_chunks",
            []
        )
    )

    expanded_count = len(
        result.get(
            "expanded_top_chunks",
            []
        )
    )

    baseline_quality = (
        "High"
        if baseline_count >= 5
        else "Medium"
        if baseline_count >= 3
        else "Low"
    )

    expanded_quality = (
        "High"
        if expanded_count >= 5
        else "Medium"
        if expanded_count >= 3
        else "Low"
    )

    comparison_table.append({

        "Question":
            result["question_id"],

        "Baseline Evidence Quality":
            baseline_quality,

        "Improved Evidence Quality":
            expanded_quality,

        "Improvement Observed":
            result.get(
                "retrieval_improvement_analysis",
                ""
            ),

        "Failure Mode":
            result.get(
                "failure_mode",
                "No significant failure observed"
            )
    })

comparison_df = pd.DataFrame(comparison_table)

comparison_df

,Question,Baseline Evidence Quality,Improved Evidence Quality,Improvement Observed,Failure Mode
0,Q1,High,High,"{'baseline_count': 1, 'expanded_count': 12, 'n...",No significant failure observed
1,Q2,High,High,"{'baseline_count': 1, 'expanded_count': 19, 'n...",No significant failure observed
2,Q3,High,High,"{'baseline_count': 1, 'expanded_count': 17, 'n...",No significant failure observed
3,Q4,High,High,"{'baseline_count': 1, 'expanded_count': 20, 'n...",No significant failure observed
